In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup + data

In [2]:
import torch, numpy as np, pandas as pd
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from datasets import Dataset
from sklearn.model_selection import train_test_split

DATA    = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OPTIONS = list("ABCDE")
device  = "cuda" if torch.cuda.is_available() else "cpu"

train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
train["label"] = train["answer"].map({c: i for i, c in enumerate(OPTIONS)})

def make_text(df):
    return (df["prompt"].astype(str) + " A: " + df["A"].astype(str)
            + " B: " + df["B"].astype(str) + " C: " + df["C"].astype(str)
            + " D: " + df["D"].astype(str) + " E: " + df["E"].astype(str))

train_text = make_text(train).tolist()
test_text  = make_text(test).tolist()
labels     = train["label"].tolist()

tr_idx, va_idx = train_test_split(range(len(train)), test_size=0.1,
                                  random_state=42, stratify=labels)

# Fine-tune function + both models train

In [3]:
def finetune(model_name, save_dir):
    tok = AutoTokenizer.from_pretrained(model_name)
    def enc(batch):
        return tok(batch["text"], truncation=True, max_length=384)
    tr_ds = Dataset.from_dict({"text": [train_text[i] for i in tr_idx],
                               "labels": [labels[i] for i in tr_idx]}).map(enc, batched=True)
    va_ds = Dataset.from_dict({"text": [train_text[i] for i in va_idx],
                               "labels": [labels[i] for i in va_idx]}).map(enc, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
    args = TrainingArguments(output_dir=save_dir, learning_rate=2e-5,
                             per_device_train_batch_size=8, num_train_epochs=2,
                             eval_strategy="epoch", save_strategy="no",
                             fp16=False, bf16=True, report_to="none", logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds,
                      eval_dataset=va_ds, processing_class=tok)
    trainer.train()
    model.save_pretrained(save_dir); tok.save_pretrained(save_dir)
    return save_dir

deberta_dir = finetune("microsoft/deberta-v3-small", "ft-deberta")
roberta_dir = finetune("roberta-base", "ft-roberta")
print("Both models fine-tuned")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along d

Epoch,Training Loss,Validation Loss
1,2.740362,1.494012
2,0.614741,0.259156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Both models fine-tuned


# Inference helpers

In [4]:
def load(save_dir):
    tok = AutoTokenizer.from_pretrained(save_dir)
    m = AutoModelForSequenceClassification.from_pretrained(save_dir).to(device).eval()
    return tok, m

tok_d, model_d = load(deberta_dir)
tok_r, model_r = load(roberta_dir)

@torch.no_grad()
def probs(texts, tok, model, bs=32):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], truncation=True, max_length=384,
                  padding=True, return_tensors="pt").to(device)
        out.append(F.softmax(model(**enc).logits, dim=-1).cpu().numpy())
    return np.vstack(out)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

**Load the fine-tuned DeBERTa and RoBERTa models.**

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

**Question 1:**

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

*(answer format : eg - A, probability of A)*

**Using the same sample (row index 25), average the class probabilities from both models.**

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

**Question 2:**

Which answer option receives the highest averaged probability after simple probability ensembling?

**Apply weighted probability averaging after Softmax using the following weights:**

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

**Question 3:**

Which answer option is ranked first after weighted ensembling?

**Using the weighted ensemble probabilities from Q3, rank all five answer options.**

Write the final prediction exactly in Kaggle submission format.

**Question 4:**

What is the Top-3 prediction string for row index 25?

Example : C A E

In [5]:
t25 = [test_text[25]]
p_d = probs(t25, tok_d, model_d)[0]      
p_r = probs(t25, tok_r, model_r)[0]

i1 = int(p_d.argmax())
Q1 = f"{OPTIONS[i1]}, {p_d[i1]:.4f}"

p_avg = (p_d + p_r) / 2
Q2 = OPTIONS[int(p_avg.argmax())]

p_w = 0.7 * p_d + 0.3 * p_r
Q3 = OPTIONS[int(p_w.argmax())]

Q4 = " ".join(OPTIONS[j] for j in np.argsort(-p_w)[:3])
print("Q1:", Q1, "| Q2:", Q2, "| Q3:", Q3, "| Q4:", Q4)

Q1: A, nan | Q2: A | Q3: A | Q4: A B C


**Run the weighted ensemble pipeline on every row of test.csv.**

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

**Question 5:**

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [6]:
P_d_test = probs(test_text, tok_d, model_d)
P_r_test = probs(test_text, tok_r, model_r)
P_w_test = 0.7 * P_d_test + 0.3 * P_r_test

order = np.argsort(-P_w_test, axis=1)
preds = [" ".join(OPTIONS[j] for j in row[:3]) for row in order]
submission = pd.DataFrame({"ID": test["id"], "Prediction": preds})
submission.to_csv("submission.csv", index=False)
Q5 = len(submission)
print("Q5:", Q5)

Q5: 500


**For the first 50 rows of test.csv, create two versions of every prompt:**

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

**Question 6:**

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [7]:
orig50 = test_text[:50]
aug50  = ["Answer the following multiple-choice question carefully: " + t for t in orig50]

P_orig = probs(orig50, tok_d, model_d)
P_aug  = probs(aug50,  tok_d, model_d)
P_tta  = (P_orig + P_aug) / 2

Q6 = int((P_orig.argmax(1) != P_tta.argmax(1)).sum())
print("Q6:", Q6)

Q6: 0


**Process the first 100 rows of test.csv. And compare the Top-1 prediction from:**

1. DeBERTa

2. Weighted Ensemble

**Question 7:**

How many rows have different Top-1 predictions?

**For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:**

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

**Question 8:**

How many rows have a positive confidence gain (greater than 0)?

**For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:**

1. DeBERTa alone

2. Weighted Ensemble

**Question 9:**

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [8]:
P_d100 = P_d_test[:100]
P_w100 = P_w_test[:100]

Q7 = int((P_d100.argmax(1) != P_w100.argmax(1)).sum())

gain = P_w100.max(1) - P_d100.max(1)
Q8 = int((gain > 0).sum())

top3_d = np.argsort(-P_d100, axis=1)[:, :3]
top3_w = np.argsort(-P_w100, axis=1)[:, :3]
Q9 = int((~(top3_d == top3_w).all(axis=1)).sum())
print("Q7:", Q7, "| Q8:", Q8, "| Q9:", Q9)

Q7: 0 | Q8: 0 | Q9: 0


**Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.**

**Question 10:**

What is the final MAP@3 score?

*(Round to 4 decimal places.)*

In [9]:
val100_idx  = va_idx[:100]
val_texts   = [train_text[i] for i in val100_idx]
val_labels  = np.array([labels[i] for i in val100_idx])

Pv_d = probs(val_texts, tok_d, model_d)
Pv_r = probs(val_texts, tok_r, model_r)
Pv_w = 0.7 * Pv_d + 0.3 * Pv_r

def map3(P, y):
    order = np.argsort(-P, axis=1); s = 0.0
    for o, l in zip(order, y):
        pos = int(np.where(o == l)[0][0])
        if pos < 3: s += 1.0 / (pos + 1)
    return s / len(y)

Q10 = round(map3(Pv_w, val_labels), 4)
print("Q10:", Q10)

Q10: 0.3833


In [10]:
print("MILESTONE 5 - ANSWERS")
for k, v in dict(Q1=Q1, Q2=Q2, Q3=Q3, Q4=Q4, Q5=Q5,
                 Q6=Q6, Q7=Q7, Q8=Q8, Q9=Q9, Q10=Q10).items():
    print(f"{k}: {v}")

MILESTONE 5 - ANSWERS
Q1: A, nan
Q2: A
Q3: A
Q4: A B C
Q5: 500
Q6: 0
Q7: 0
Q8: 0
Q9: 0
Q10: 0.3833
